In [13]:
# =============================================================================
# GPM IMERG GIOVANNI CSV PROCESSOR
# Converts Half-hourly IMERG (mm) to:
#   1. Half-hour Rainfall (mm) - already in data
#   2. Hourly Rainfall (mm)
#   3. Daily Rainfall (mm)
#
# Author : Jintu Moni Bhuyan
# Compatible with Pandas 2.x
# =============================================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =============================================================================
# INPUT CSV
# =============================================================================

csv_file = r"D:\NERDRR\July 2026\Nagaland_Jorhat\Nagaland\Giovanni\g4.areaAvgTimeSeries.GPM_3IMERGHHL_07_precipitation.20260717-20260720.94E_26N_95E_27N.csv"

# =============================================================================
# OUTPUT DIRECTORY
# =============================================================================

output_dir = os.path.join(os.path.dirname(csv_file), "Output")
os.makedirs(output_dir, exist_ok=True)

# =============================================================================
# FIND DATA HEADER
# =============================================================================

print("Searching data table...")

with open(csv_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

header_row = None

for i, line in enumerate(lines):
    if line.lower().startswith("time"):
        header_row = i
        break

if header_row is None:
    raise Exception("Could not find the data table.")

print(f"Header found at line : {header_row}")

# =============================================================================
# READ CSV
# =============================================================================

try:
    df = pd.read_csv(csv_file, skiprows=header_row)
except:
    df = pd.read_csv(csv_file, skiprows=header_row, sep="\t")

print(df.head())

# =============================================================================
# DETECT COLUMNS
# =============================================================================

time_col = None
rain_col = None

for col in df.columns:

    col_lower = col.lower().strip()

    if "time" in col_lower:
        time_col = col

    if "mean" in col_lower or "precip" in col_lower:
        rain_col = col

print("Time column :", time_col)
print("Rain column :", rain_col)

if time_col is None:
    raise Exception("Time column not found.")

if rain_col is None:
    raise Exception("Rainfall column not found.")

# =============================================================================
# KEEP REQUIRED COLUMNS
# =============================================================================

df = df[[time_col, rain_col]].copy()

df.columns = ["Datetime", "Rainfall_30min_mm"]  # Data is already in mm per half-hour

# =============================================================================
# CONVERT DATETIME
# =============================================================================

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

df = df.dropna(subset=["Datetime"])

# =============================================================================
# CONVERT TO NUMERIC
# =============================================================================

df["Rainfall_30min_mm"] = pd.to_numeric(df["Rainfall_30min_mm"], errors="coerce")

df = df.dropna(subset=["Rainfall_30min_mm"])

# =============================================================================
# REMOVE FILL VALUES
# =============================================================================

df = df[df["Rainfall_30min_mm"] > -9990]

# =============================================================================
# SORT
# =============================================================================

df = df.sort_values("Datetime").reset_index(drop=True)

# =============================================================================
# EXTRACT DATE RANGE FOR TITLES
# =============================================================================

start_date = df["Datetime"].min().strftime("%d %B %Y")
end_date = df["Datetime"].max().strftime("%d %B %Y")
date_range = f"{start_date} to {end_date}"

# Also create shorter version for filenames if needed
date_range_short = f"{df['Datetime'].min().strftime('%Y%m%d')}-{df['Datetime'].max().strftime('%Y%m%d')}"

print(f"Date range: {date_range}")

# =============================================================================
# SAVE HALF-HOURLY CSV
# =============================================================================

halfhour_csv = os.path.join(output_dir, "HalfHourly_Rainfall.csv")

df.to_csv(halfhour_csv, index=False)

print("Saved :", halfhour_csv)

# =============================================================================
# HOURLY RAINFALL
# =============================================================================

hourly = (
    df.set_index("Datetime")["Rainfall_30min_mm"]
      .resample("1h")
      .sum()
      .reset_index()
)

hourly.columns = ["Datetime", "Hourly_Rainfall_mm"]

hourly_csv = os.path.join(output_dir, "Hourly_Rainfall.csv")

hourly.to_csv(hourly_csv, index=False)

print("Saved :", hourly_csv)

# =============================================================================
# DAILY RAINFALL
# =============================================================================

daily = (
    hourly.set_index("Datetime")["Hourly_Rainfall_mm"]
          .resample("1d")
          .sum()
          .reset_index()
)

daily.columns = ["Date", "Daily_Rainfall_mm"]

daily_csv = os.path.join(output_dir, "Daily_Rainfall.csv")

daily.to_csv(daily_csv, index=False)

print("Saved :", daily_csv)

# =============================================================================
# PLOTTING FUNCTION
# =============================================================================

plt.rcParams["font.family"] = "Times New Roman"

def plot_rainfall(df,
                  xcol,
                  ycol,
                  title_prefix,
                  ylabel,
                  outfile,
                  date_range_str,
                  source="GPM IMERG Multi-Satellite Data"):

    fig, ax = plt.subplots(figsize=(14,6))

    # Filled area
    ax.fill_between(
        df[xcol],
        df[ycol],
        color="#8fb3d1",
        alpha=0.65
    )

    # Line
    ax.plot(
        df[xcol],
        df[ycol],
        color="#1f77b4",
        linewidth=3.5,
        marker="o",
        markersize=10
    )

    # Value labels - only add if not too many points
    if len(df) <= 30:
        for x, y in zip(df[xcol], df[ycol]):
            ax.text(
                x,
                y + max(df[ycol]) * 0.015,
                f"{y:.2f}",
                ha="center",
                fontsize=10,
                fontweight="bold"
            )

    # Source box
    ax.text(
        0.02,
        0.97,
        source,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            linewidth=1.5
        )
    )

    # Title with automatic date range
    title = f"{title_prefix} ({date_range_str})"
    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=20
    )

    ax.set_xlabel(
        "Date",
        fontsize=15,
        fontweight="bold"
    )

    ax.set_ylabel(
        ylabel,
        fontsize=15,
        fontweight="bold"
    )

    # Grid
    ax.grid(
        True,
        linestyle="--",
        linewidth=1,
        alpha=0.5
    )

    # Thick border
    for s in ax.spines.values():
        s.set_linewidth(1.5)

    # Tick formatting - adjust based on number of points
    if len(df) > 20:
        # For half-hourly data - show fewer ticks to avoid overlap
        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%d-%b\n%H:%M")
        )
        # Set major locator to show every 4th tick for half-hourly data
        if len(df) > 48:  # More than 24 hours of half-hourly data
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
        else:
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    else:
        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%d-%b-%Y")
        )

    # Rotate labels and adjust spacing
    plt.xticks(rotation=45, ha='right')

    # Adjust layout to prevent label cutoff
    plt.tight_layout()

    plt.savefig(outfile, dpi=600, bbox_inches='tight')

    plt.close()

# =============================================================================
# PLOT HALF-HOURLY
# =============================================================================

plot_rainfall(
    df,
    "Datetime",
    "Rainfall_30min_mm",
    "Half-hourly Area-Averaged Rainfall",
    "Rainfall (mm)",
    os.path.join(output_dir, "HalfHourly_Rainfall.png"),
    date_range
)

# =============================================================================
# PLOT HOURLY
# =============================================================================

plot_rainfall(
    hourly,
    "Datetime",
    "Hourly_Rainfall_mm",
    "Hourly Area-Averaged Rainfall",
    "Rainfall (mm)",
    os.path.join(output_dir, "Hourly_Rainfall.png"),
    date_range
)

# =============================================================================
# PLOT DAILY
# =============================================================================

plot_rainfall(
    daily,
    "Date",
    "Daily_Rainfall_mm",
    "Daily Area-Averaged Rainfall",
    "Daily Rainfall (mm)",
    os.path.join(output_dir, "Daily_Rainfall.png"),
    date_range
)

# =============================================================================
# SUMMARY
# =============================================================================

print("\n")
print("=" * 70)
print("PROCESS COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"Half-hour records : {len(df)}")
print(f"Hourly records    : {len(hourly)}")
print(f"Daily records     : {len(daily)}")
print(f"Date range        : {date_range}")

print("\nOutput folder:")
print(output_dir)

print("=" * 70)

Searching data table...
Header found at line : 8
                  time   mean_GPM_3IMERGHHL_07_precipitation
0  2026-07-17 00:00:00                              0.564951
1  2026-07-17 00:30:00                              0.143208
2  2026-07-17 01:00:00                              0.214043
3  2026-07-17 01:30:00                              0.052975
4  2026-07-17 02:00:00                              0.082658
Time column : time
Rain column :  mean_GPM_3IMERGHHL_07_precipitation
Date range: 17 July 2026 to 19 July 2026
Saved : D:\NERDRR\July 2026\Nagaland_Jorhat\Nagaland\Giovanni\Output\HalfHourly_Rainfall.csv
Saved : D:\NERDRR\July 2026\Nagaland_Jorhat\Nagaland\Giovanni\Output\Hourly_Rainfall.csv
Saved : D:\NERDRR\July 2026\Nagaland_Jorhat\Nagaland\Giovanni\Output\Daily_Rainfall.csv


C:\Users\Standard\AppData\Local\Temp\ipykernel_11540\1684268209.py:176: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  .resample("1d")




PROCESS COMPLETED SUCCESSFULLY
Half-hour records : 124
Hourly records    : 62
Daily records     : 3
Date range        : 17 July 2026 to 19 July 2026

Output folder:
D:\NERDRR\July 2026\Nagaland_Jorhat\Nagaland\Giovanni\Output
